In [1]:
import os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
DATE_START = pd.Timestamp("2022-07-01 00:00:00")
DATE_END   = pd.Timestamp("2025-06-30 23:00:00")

datasets = {
    "CPCB":    "/home/rishi/ML Projects/Air Pollution/CPCB/separated",
    "EPA":     "/home/rishi/ML Projects/Air Pollution/EPA/epa_data_by_site_all_years",
    "AURN":    "/home/rishi/ML Projects/Air Pollution/AURN/aurn_processed",
    "EEA_FR":  "/home/rishi/ML Projects/Air Pollution/EEA/FR/processed",
    "EEA_DE":  "/home/rishi/ML Projects/Air Pollution/EEA/DE/processed",
    "SINAICA": "/home/rishi/ML Projects/Air Pollution/SINAICA/processed",
    "CNEMC":   "/home/rishi/ML Projects/Air Pollution/CNEMC/separated",
}

In [3]:
def analyse_file(filepath, date_start, date_end):
    """Return per-file stats including missingness and max contiguous gap."""
    try:
        df = pd.read_csv(filepath, parse_dates=["Timestamp"])
    except (ValueError, KeyError):
        return None

    pollutant_col = df.columns[1]

    # Reindex to full hourly range to match visualise.py / imputation.py
    full_index = pd.date_range(date_start, date_end, freq="h")
    series = df.set_index("Timestamp")[pollutant_col].reindex(full_index)

    # Match imputation.py: treat negatives as NaN
    series = series.where(series >= 0)

    total = len(series)
    nan_count = int(series.isna().sum())
    missingness = round(100 * nan_count / total, 4)

    # Max contiguous gap (in hours)
    is_nan = series.isna().values
    max_gap = 0
    current_gap = 0
    for v in is_nan:
        if v:
            current_gap += 1
            if current_gap > max_gap:
                max_gap = current_gap
        else:
            current_gap = 0

    return {
        "file": filepath.name,
        "total_rows": total,
        "nan_count": nan_count,
        "missingness_pct": missingness,
        "max_gap_hours": max_gap,
    }


def analyse_dataset(directory, date_start, date_end, max_workers=24):
    """Analyse all CSVs in a directory and return a list of dicts."""
    csv_files = sorted(Path(directory).glob("*.csv"))
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(analyse_file, f, date_start, date_end): f for f in csv_files}
        for future in tqdm(as_completed(futures), total=len(futures), desc=Path(directory).name, leave=False):
            res = future.result()
            if res is not None:
                results.append(res)
    return results

In [4]:
# Build a per-file missingness dataframe for every dataset
all_rows = []

for name, directory in datasets.items():
    if not os.path.isdir(directory):
        print(f"[SKIP] {name}: {directory}")
        continue
    print(f"Processing {name} ...")
    file_results = analyse_dataset(directory, DATE_START, DATE_END)
    for r in file_results:
        r["dataset"] = name
    all_rows.extend(file_results)
    print(f"  {len(file_results)} files analysed")

df_miss = pd.DataFrame(all_rows)[["dataset", "file", "total_rows", "nan_count", "missingness_pct", "max_gap_hours"]]
print(f"\nTotal files: {len(df_miss)}")
df_miss.head(10)

Processing CPCB ...


separated:   0%|          | 0/3323 [00:00<?, ?it/s]

  3323 files analysed
Processing EPA ...


epa_data_by_site_all_years:   0%|          | 0/4020 [00:00<?, ?it/s]

  4020 files analysed
Processing AURN ...


...

processed:   0%|          | 0/890 [00:00<?, ?it/s]

  889 files analysed
Processing CNEMC ...


separated:   0%|          | 0/10368 [00:00<?, ?it/s]

  10368 files analysed

Total files: 22388


,dataset,file,total_rows,nan_count,missingness_pct,max_gap_hours
0,CPCB,site_107_Pusa_Delhi_IMD_Ozone.csv,26304,1559,5.9269,798
1,CPCB,site_106_IGI_Airport_(T3)_Delhi_IMD_PM2.5.csv,26304,911,3.4634,586
2,CPCB,site_109_Lodhi_Road_Delhi_IMD_CO.csv,26304,333,1.2660,82
3,CPCB,site_103_CRRI_Mathura_Road_Delhi_IMD_NO2.csv,26304,3029,11.5154,1570
4,CPCB,site_105_North_Campus_DU_Delhi_IMD_CO.csv,26304,915,3.4786,64
5,CPCB,site_104_Burari_Crossing_Delhi_IMD_CO.csv,26304,1818,6.9115,791
6,CPCB,site_103_CRRI_Mathura_Road_Delhi_IMD_Ozone.csv,26304,2750,10.4547,1570
7,CPCB,site_106_IGI_Airport_(T3)_Delhi_IMD_NO2.csv,26304,915,3.4786,586
8,CPCB,site_104_Burari_Crossing_Delhi_IMD_PM10.csv,26304,3857,14.6632,1001
9,CPCB,site_105_North_Campus_DU_Delhi_IMD_PM10.csv,26304,979,3.7219,64


In [ ]:
# Filter files into missingness buckets with scaled max gap thresholds
# Linear scale: 5% -> 72 hrs (3 days), 30% -> 336 hrs (14 days)
# Uses <= to match imputation.py logic (missing_pct > thresh rejects, max_gap <= thresh keeps)
def max_gap_for_bucket(pct):
    return 72 + (336 - 72) * (pct - 5) / (30 - 5)

buckets = {
    "lt_5pct":  ( 5, max_gap_for_bucket(5)),   # 72 hrs
    "lt_10pct": (10, max_gap_for_bucket(10)),   # 124.8 hrs
    "lt_15pct": (15, max_gap_for_bucket(15)),   # 177.6 hrs
    "lt_20pct": (20, max_gap_for_bucket(20)),   # 230.4 hrs
    "lt_30pct": (30, max_gap_for_bucket(30)),   # 336 hrs
}

print("Bucket thresholds:")
for name, (pct, gap) in buckets.items():
    print(f"  {name}: missingness <= {pct}%, max_gap <= {gap:.0f} hrs ({gap/24:.1f} days)")
print()

base_dir = Path("/home/rishi/ML Projects/Air Pollution Bench/Air-Pollution-Bench/imputation_ablation")
base_dir.mkdir(exist_ok=True)

for ds_name in df_miss["dataset"].unique():
    ds_dir = base_dir / ds_name
    ds_dir.mkdir(exist_ok=True)

    ds_df = df_miss[df_miss["dataset"] == ds_name]

    for bucket_name, (miss_thresh, gap_thresh) in buckets.items():
        filtered = ds_df[(ds_df["missingness_pct"] <= miss_thresh) & (ds_df["max_gap_hours"] <= gap_thresh)]
        out_df = filtered[["file", "missingness_pct", "max_gap_hours"]].sort_values("missingness_pct").reset_index(drop=True)
        out_path = ds_dir / f"{bucket_name}.csv"
        out_df.to_csv(out_path, index=False)
        print(f"{ds_name}/{bucket_name}: {len(out_df)} files")

print(f"\nAll bucket CSVs written to {base_dir}")

Bucket thresholds:
  lt_5pct: missingness <= 5%, max_gap <= 72 hrs (3.0 days)
  lt_10pct: missingness <= 10%, max_gap <= 125 hrs (5.2 days)
  lt_15pct: missingness <= 15%, max_gap <= 178 hrs (7.4 days)
  lt_20pct: missingness <= 20%, max_gap <= 230 hrs (9.6 days)
  lt_30pct: missingness <= 30%, max_gap <= 336 hrs (14.0 days)

CPCB/lt_5pct: 202 files
CPCB/lt_10pct: 497 files
CPCB/lt_15pct: 832 files
CPCB/lt_20pct: 918 files
CPCB/lt_30pct: 1104 files
EPA/lt_5pct: 396 files
EPA/lt_10pct: 823 files
EPA/lt_15pct: 1189 files
EPA/lt_20pct: 1436 files
EPA/lt_30pct: 1667 files
AURN/lt_5pct: 86 files
AURN/lt_10pct: 132 files
AURN/lt_15pct: 190 files
AURN/lt_20pct: 220 files
AURN/lt_30pct: 259 files
EEA_FR/lt_5pct: 224 files
EEA_FR/lt_10pct: 435 files
EEA_FR/lt_15pct: 603 files
EEA_FR/lt_20pct: 690 files
EEA_FR/lt_30pct: 807 files
EEA_DE/lt_5pct: 940 files
EEA_DE/lt_10pct: 1180 files
EEA_DE/lt_15pct: 1249 files
EEA_DE/lt_20pct: 1291 files
EEA_DE/lt_30pct: 1334 files
SINAICA/lt_5pct: 1 files
SINAI

In [6]:
# Verify lt_30pct matches imputation.py's get_sites_per_pollutant logic
# Replicate imputation.py exactly: read vis_dicts, replace negatives, check missingness & max_gap

dicts_dirs = {
    "CPCB":    "/home/rishi/ML Projects/Air Pollution/CPCB/vis_dicts",
    "EPA":     "/home/rishi/ML Projects/Air Pollution/EPA/visualize_dicts",
    "AURN":    "/home/rishi/ML Projects/Air Pollution/AURN/vis_dicts",
    "EEA_FR":  "/home/rishi/ML Projects/Air Pollution/EEA/FR/vis_dicts",
    "EEA_DE":  "/home/rishi/ML Projects/Air Pollution/EEA/DE/vis_dicts",
    "SINAICA": "/home/rishi/ML Projects/Air Pollution/SINAICA/vis_dicts",
    "CNEMC":   "/home/rishi/ML Projects/Air Pollution/CNEMC/vis_dicts",
}

features = ["PM2.5 (µg/m³)", "PM10 (µg/m³)", "NO2 (µg/m³)", "SO2 (µg/m³)", "CO (mg/m³)", "Ozone (µg/m³)"]
MAX_DATA_MISSING = 30
MAX_GAP_HOURS = 336

all_match = True
for ds_name, dicts_dir in dicts_dirs.items():
    if not os.path.isdir(dicts_dir):
        print(f"[SKIP] {ds_name}: {dicts_dir} not found")
        continue

    # Collect valid files from imputation.py logic
    # vis_dicts column names already end in .csv (e.g. "site_103_..._CO.csv")
    imputation_valid_files = set()
    for pol in features:
        safe_key = pol.split(" ")[0]
        dict_path = os.path.join(dicts_dir, f"{safe_key}_df.csv")
        if not os.path.exists(dict_path):
            continue
        df = pd.read_csv(dict_path, index_col=0, parse_dates=True)
        df = df.where(df >= 0)  # replace_negatives_with_nan

        missing_pct = df.isnull().sum(axis=0) * 100 / len(df)

        for col in df.columns:
            if missing_pct[col] > MAX_DATA_MISSING:
                continue
            is_missing = df[col].isnull().values
            max_gap = cur = 0
            for m in is_missing:
                if m:
                    cur += 1
                    max_gap = max(max_gap, cur)
                else:
                    cur = 0
            if max_gap <= MAX_GAP_HOURS:
                # col already ends in .csv, use as-is
                imputation_valid_files.add(col)

    # Get our lt_30pct files for this dataset
    our_30 = set(df_miss[
        (df_miss["dataset"] == ds_name) &
        (df_miss["missingness_pct"] <= 30) &
        (df_miss["max_gap_hours"] <= 336)
    ]["file"].values)

    only_in_imputation = imputation_valid_files - our_30
    only_in_ours = our_30 - imputation_valid_files

    if not only_in_imputation and not only_in_ours:
        print(f"  {ds_name}: MATCH ({len(our_30)} files)")
    else:
        all_match = False
        print(f"  {ds_name}: MISMATCH — imputation={len(imputation_valid_files)}, ours={len(our_30)}")
        if only_in_imputation:
            examples = sorted(only_in_imputation)[:5]
            print(f"    only in imputation.py ({len(only_in_imputation)}): {examples}...")
        if only_in_ours:
            examples = sorted(only_in_ours)[:5]
            print(f"    only in ours ({len(only_in_ours)}): {examples}...")

print(f"\n{'ALL DATASETS MATCH' if all_match else 'SOME MISMATCHES FOUND'}")

  CPCB: MATCH (1104 files)
  EPA: MATCH (1667 files)
  AURN: MATCH (259 files)
  EEA_FR: MATCH (807 files)
  EEA_DE: MATCH (1334 files)
  SINAICA: MATCH (75 files)
  CNEMC: MATCH (8893 files)

ALL DATASETS MATCH
